# 02a - Competitor Keyword Scraping (Selenium)
Extraction and cleaning of commercial tourist destinations using Selenium. This process gathers competitor data to establish target keywords for the Google Ads Planner.

### Legal & Ethical Notice

This notebook scrapes publicly accessible web pages to extract 
destination keyword data for academic research purposes.

**Scope:** Only public destination listing pages are accessed.  
No authentication, personal data, or paywalled content is involved.

**robots.txt:** Verified on 2026-06-03. The scraped URL (`/es-es/viajes`)  
is not restricted by the site's crawling policy (`User-agent: *` only  
disallows `/cdn-cgi/` and `/api/`).

**Rate limiting:** Random delays (`time.sleep(random.uniform())`)  
are implemented to avoid server overload.

**Purpose:** Academic final project and portfolio.

## 1. Imports

In [12]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from pathlib import Path
from unidecode import unidecode

import pandas as pd
import time
import random

## 2. Paths configuration

In [13]:
INTERIM_PATH_RAW = Path("../data/interim/raw")

INTERIM_PATH_RAW.mkdir(
    parents=True,
    exist_ok=True
)

## 3. Scraping Configuration

### Exclusion Criteria
These terms were identified during manual review of the raw scraping output.
They are excluded because the scraper captured them as false positives:
navigation links, legal pages, product categories, contact info and audience 
segments that appear as anchor tags (`<a>`) in the website's HTML but do not 
represent searchable geographic destinations.

Exclusion was applied iteratively after inspecting the raw Excel export.

> **Note:** product+destination hybrids like `"cruceros por egipto"` are 
> excluded because `"cruceros"` is a product category, not a destination 
> signal. The destination (`"egipto"`) is captured separately via the 
> INE country master list.

In [14]:
URL = "https://www.azulmarino.com/es-es/viajes"

EXCLUDED = {

    "politica",
    "cookies",
    "quienes somos",
    "contacto",
    "newsletter",
    "telefono",
    "blog",
    "agencias",
    "empleo",
    "actividades",
    "910 013 800",
    "910013800",
    "aviso legal",
    "bases legales",
    "booking@azulmarino.com",
    "circuitos",
    "concierto + viaje",
    "condiciones de pago",
    "corporate travel",
    "cruceros",
    "cruceros por egipto",
    "destinos",
    "destinos exoticos",
    "directorio de agencias de viajes azulmarino",
    "financia tu viaje",
    "franquicias especializadas",
    "home",
    "imprescindibles",
    "mayores de 55",
    "musica + viaje",
    "novios",
    "politica de cookies",
    "politica de privacidad",
    "regatas rcnp",
    "seguros",
    "solo hotel",
    "trabaja con nosotros",
    "ver todos nuestros destinos",
    "viajes con expertos",
    "viajes de estudiantes",
    "viajes de expedicion",
    "viajes de novios",
    "viajes en grupo",
    "viajes unicos",
    "vuelo + hotel",
    "calidades caribe"
}

## 4. Core Scraping Methods

In [15]:
def normalize_text(text):
    """
    Standardizes text by converting to lowercase, removing accents, and stripping whitespace.
    Args: text (str): The raw text string to normalize.
    Returns: str or None: The cleaned string, or None if the input is missing (NaN).
    """

    if pd.isna(text):
        return None

    text = str(text).lower().strip()
    text = unidecode(text)
    text = text.replace("-", " ")

    return text

In [ ]:
def create_driver():
    """
    Initializes and configures the Selenium Chrome WebDriver.
    
    Uses ChromeDriverManager to automatically download and link the correct 
    driver version for the local OS.

    Returns: webdriver.Chrome: An active Selenium WebDriver instance.
    """
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    return driver

In [17]:
def reject_cookies(driver):
    """
    Waits for the cookies popup and dismisses it.
    
    Implements an Explicit Wait (max 10 seconds) to ensure the button is clickable,
    followed by a random human-like delay to bypass basic anti-bot detection.

    Args: driver (webdriver.Chrome): The active Selenium WebDriver instance.
    """
    try:
        cookies_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.ID, "didomi-notice-disagree-button"))
        )
        time.sleep(random.uniform(0.5, 1.5)) # EC expected conditions

        cookies_button.click()
        print("✔ Cookies rejected successfully")

    except Exception as e:
        print(f"Cookie popup not found or timed out. Details: {e}")

In [18]:
def scroll_page(driver):
    """
    Executes a JavaScript command to scroll to the bottom of the page.
    Args: driver (webdriver.Chrome): The active Selenium WebDriver instance.
    """
    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight);"
    )

    time.sleep(5)

In [19]:
def extract_links(driver):
    """
    Locates and extracts all anchor HTML tags (<a>) from the current page.
    Args: driver (webdriver.Chrome): The active Selenium WebDriver instance.
    Returns: list: A list of Selenium web elements representing the links.
    """
    links = driver.find_elements(
        By.TAG_NAME,
        "a"
    )

    return links

In [ ]:
def process_links(links):
    """
    Iterates through raw web links, cleans the text, and filters out excluded terms.
    Args: links (list): A list of Selenium anchor elements.
    Returns: list: A list of dictionaries containing the processed destination data.
    """
    results = []

    for link in links:
        text = link.text.strip()
        href = link.get_attribute("href")

        if not text or not href:
            continue
        
        text_clean = normalize_text(text)

        if len(text_clean) < 3:
            continue

        if any(term in text_clean for term in EXCLUDED):
            continue

        results.append({
            "raw_title": text,
            "search_term": text_clean,
            "parent_country": None,
            "keyword_type": None,
            "source": "azulmarino",
            "url": href
        })


    return results

In [21]:
def build_dataframe(results):
    """
    Converts the processed link dictionaries into a pandas DataFrame.
    Args: results (list): The list of parsed link dictionaries.
    Returns: df: sorted DataFrame ready for export.
    """
    df = (
        pd.DataFrame(results)
        .drop_duplicates()
        .sort_values("search_term")
        .reset_index(drop=True)
    )

    return df

## 5. Scraper Orchestration

In [22]:
def main_scraping():
    """
    Orchestrates the complete web scraping workflow.
       
    Initializes the driver, navigates to the target URL, handles cookies,
    scrolls, extracts data and safely closes the browser.

    Returns: df wit the compiled dataset of destination keywords.
        """
    driver = create_driver()
    driver.get(URL)
    
    time.sleep(5)
    print(f"Page Title: {driver.title}")
    
    reject_cookies(driver)
    scroll_page(driver)
    
    links = extract_links(driver)
    results = process_links(links)
    
    driver.quit()
    
    df = build_dataframe(results)

    return df

## 6. Job Execution

In [23]:
df_scraping = main_scraping()
df_scraping.head()

Page Title: Viajes a los destinos Azulmarino | Agencia de Viajes Azulmarino
✔ Cookies rejected successfully


,raw_title,search_term,parent_country,keyword_type,source,url
0,Albania,albania,None,None,azulmarino,https://www.azulmarino.com/viajes/europa/albania
1,Alemania,alemania,None,None,azulmarino,https://www.azulmarino.com/viajes/europa/alemania
2,Alpes,alpes,None,None,azulmarino,https://www.azulmarino.com/viajes/europa/alpes
3,Andorra,andorra,None,None,azulmarino,https://www.azulmarino.com/viajes/europa/andorra
4,Angola,angola,None,None,azulmarino,https://www.azulmarino.com/viajes/africa/angola


## 7. Data Validation

In [24]:
print(f"Dataset Shape: {df_scraping.shape}\n")

print("--- NULL & EMPTY VALUES ---")
print(df_scraping.isnull().sum())
print(f"\nEmpty 'search_term' strings: {(df_scraping['search_term'] == '').sum()}")

print("\n--- DUPLICATES ---")
print(f"Duplicate URLs: {df_scraping.duplicated(subset=['url']).sum()}")
print(f"Duplicate Search Terms: {df_scraping.duplicated(subset=['search_term']).sum()}")

print("\n--- SUMMARY ---")
print(f"Total unique keywords extracted: {df_scraping['search_term'].nunique()}")

Dataset Shape: (223, 6)

--- NULL & EMPTY VALUES ---
raw_title           0
search_term         0
parent_country    223
keyword_type      223
source              0
url                 0
dtype: int64

Empty 'search_term' strings: 0

--- DUPLICATES ---
Duplicate URLs: 0
Duplicate Search Terms: 1

--- SUMMARY ---
Total unique keywords extracted: 222


## 8. Dataset Export

In [ ]:
df_scraping.to_parquet(
    INTERIM_PATH_RAW / "02a_azulmarino_scraped_raw.parquet"
)

# Export to Excel (For manual review)
df_scraping.to_excel(
    INTERIM_PATH_RAW / "02a_azulmarino_scraped_raw.xlsx",
    index=False
)

## 9. Manual Enrichment Guidelines
The exported Excel file requires manual data enrichment before the next stage. 
It is necessary to review the file and perform the following tasks:

* Remove any remaining irrelevant commercial terms.
* Standardize commercial names if necessary.
* Assign the correct `parent_country`.
* Define the `keyword_type` for each entry (country / region / destination brand).
* Validate commercial tourist keywords.

Save the fully reviewed file in:
`../data/interim/reviewed/azulmarino_destinations_reviewed.xlsx`
`../data/interim/reviewed/tempsdoci_destinations_reviewed.xlsx` (notebook 2b)